![Fraud detection image](cover_image.jpg)

🏦 Banks are battling frauds with machine learning models, but changing data patterns can weaken these defenses. London's Poundbank needs your help to figure out why their fraud detection models aren't as accurate anymore.

Poundbank recommends the `nannyml` library for monitoring machine learning models, which is also their tool of choice.

## The data

They have provided you with a reference(test data) and analysis set(production data). A summary and preview are provided below.

## reference.csv and analysis.csv

| Column     | Description              |
|------------|--------------------------|
| `'timestamp'` | Date of the transaction. |
| `'time_since_login_min'` | Time since the user logged in to the app. |
| `'transaction_amount'` | The amount of Pounds(£) that users sent to another account. |
| `'transaction_type'` | Transaction type: <ul><li>`CASH-OUT` - Withdrawing money from an account.</li><li>`PAYMENT` - Transaction where a payment is made to a third party.</li><li>`CASH-IN` - This is the opposite of a cash-out. It involves depositing money into an account.</li><li>`TRANSFER` - Transaction which involves moving funds from one account to another.</li> |
| `'is_first_transaction'` | A binary indicator denoting if the transaction is the user's first (1 for the first transaction, 0 otherwise). |
| `'user_tenure_months'` | The duration in months since the user's account was created or since they became a member. |
| `'is_fraud'` | A binary label indicating whether the transaction is fraudulent (1 for fraud, 0 otherwise). |
| `'predicted_fraud_proba'` | The probability assigned by a detection model indicates the likelihood of a fraudulent transaction. |
| `'predicted_fraud'` |  The predicted classification label is calculated based on predicted fraud probability by the detection model (1 for predicted fraud, 0 otherwise). |

In [41]:
# Re-run this cell to install nannyml
!pip install nannyml

Defaulting to user installation because normal site-packages is not writeable


In [42]:
# Re-run this cell
# Import required libraries
import pandas as pd
import nannyml as nml
nml.disable_usage_logging()

reference = pd.read_csv("reference.csv")
analysis = pd.read_csv("analysis.csv")
reference.head()

,timestamp,time_since_login_min,transaction_amount,transaction_type,is_first_transaction,user_tenure_months,is_fraud,predicted_fraud_proba,predicted_fraud
0,2018-01-01 00:00:00.000,1.561750,3981.1,PAYMENT,False,0.318980,1.0,0.99,1
1,2018-01-01 00:08:43.152,1.658074,1267.9,PAYMENT,False,7.391323,0.0,0.07,0
2,2018-01-01 00:17:26.304,2.454287,1984.7,CASH-IN,False,0.781225,1.0,1.00,1
3,2018-01-01 00:26:09.456,2.392085,2265.2,CASH-OUT,False,0.680473,1.0,0.98,1
4,2018-01-01 00:34:52.608,2.189806,2126.8,CASH-IN,False,8.542895,1.0,0.99,1


In [43]:
# Start coding here

# Estimated Accuracy (CBPE)
estimator = nml.CBPE(
    y_pred_proba='predicted_fraud_proba',
    y_pred='predicted_fraud',
    y_true='is_fraud',
    problem_type='classification_binary',
    metrics=['accuracy'],
    timestamp_column_name='timestamp',
    chunk_period='M',
)
estimator.fit(reference)
est_results = estimator.estimate(analysis)

est_df = est_results.filter(period='analysis').to_df()
est_df.columns = ['_'.join(c).strip('_') for c in est_df.columns]
print(est_df[['chunk_key','accuracy_value','accuracy_upper_threshold','accuracy_lower_threshold','accuracy_alert']].to_string(index=False))

chunk_key  accuracy_value  accuracy_upper_threshold  accuracy_lower_threshold  accuracy_alert
  2018-11        0.941947                  0.950808                  0.936367           False
  2018-12        0.943251                  0.950808                  0.936367           False
  2019-01        0.945452                  0.950808                  0.936367           False
  2019-02        0.944608                  0.950808                  0.936367           False
  2019-03        0.943735                  0.950808                  0.936367           False
  2019-04        0.911049                  0.950808                  0.936367            True
  2019-05        0.910242                  0.950808                  0.936367            True
  2019-06        0.911663                  0.950808                  0.936367            True


In [44]:
# Realized Accuracy
calc = nml.PerformanceCalculator(
    y_pred_proba='predicted_fraud_proba',
    y_pred='predicted_fraud',
    y_true='is_fraud',
    problem_type='classification_binary',
    metrics=['accuracy'],
    timestamp_column_name='timestamp',
    chunk_period='M',
)
calc.fit(reference)
real_results = calc.calculate(analysis)

real_df = real_results.filter(period='analysis').to_df()
real_df.columns = ['_'.join(c).strip('_') for c in real_df.columns]
print(real_df[['chunk_key','accuracy_value','accuracy_upper_threshold','accuracy_lower_threshold','accuracy_alert']].to_string(index=False))

chunk_key  accuracy_value  accuracy_upper_threshold  accuracy_lower_threshold  accuracy_alert
  2018-11        0.942684                  0.950808                  0.936367           False
  2018-12        0.941004                  0.950808                  0.936367           False
  2019-01        0.951758                  0.950808                  0.936367            True
  2019-02        0.943123                  0.950808                  0.936367           False
  2019-03        0.940039                  0.950808                  0.936367           False
  2019-04        0.914632                  0.950808                  0.936367            True
  2019-05        0.915413                  0.950808                  0.936367            True
  2019-06        0.914228                  0.950808                  0.936367            True


In [45]:
# months_with_performance_alerts
month_map = {
    '01':'january','02':'february','03':'march','04':'april',
    '05':'may','06':'june','07':'july','08':'august',
    '09':'september','10':'october','11':'november','12':'december'
}

def fmt_month(key):
    year, mon = key.split('-')
    return f"{month_map[mon]}_{year}"

est_alerts  = set(est_df[est_df['accuracy_alert']  == True]['chunk_key'].tolist())
real_alerts = set(real_df[real_df['accuracy_alert'] == True]['chunk_key'].tolist())

both_alerts = est_alerts.intersection(real_alerts)

months_with_performance_alerts = [fmt_month(m) for m in sorted(both_alerts)]
print("months_with_performance_alerts =", months_with_performance_alerts)


months_with_performance_alerts = ['april_2019', 'may_2019', 'june_2019']


In [46]:
# Univariate Drift (KS + Chi-square) 
feature_cols = ['time_since_login_min', 'transaction_amount',
                'user_tenure_months', 'transaction_type', 'is_first_transaction']

drift_calc = nml.UnivariateDriftCalculator(
    column_names=feature_cols,
    timestamp_column_name='timestamp',
    chunk_period='M',
    continuous_methods=['kolmogorov_smirnov'],
    categorical_methods=['chi2'],
)
drift_calc.fit(reference)
drift_results = drift_calc.calculate(analysis)

drift_df = drift_results.filter(period='analysis').to_df()
drift_df.columns = ['_'.join(c).strip('_') for c in drift_df.columns]

alert_cols = {
    'time_since_login_min': 'time_since_login_min_kolmogorov_smirnov_alert',
    'transaction_amount':   'transaction_amount_kolmogorov_smirnov_alert',
    'user_tenure_months':   'user_tenure_months_kolmogorov_smirnov_alert',
    'is_first_transaction': 'is_first_transaction_chi2_alert',
    'transaction_type':     'transaction_type_chi2_alert',
}

# Count alerts di periode accuracy drop (Apr–Jun 2019)
critical = drift_df[drift_df['chunk_chunk_key'].isin(['2019-04','2019-05','2019-06'])]
alert_scores = {feat: critical[acol].sum() for feat, acol in alert_cols.items()}

highest_correlation_feature = max(alert_scores, key=alert_scores.get)
print("highest_correlation_feature =", repr(highest_correlation_feature))

highest_correlation_feature = 'time_since_login_min'


In [47]:
# Monthly Avg Transaction Amount Alert
avg_calc = nml.SummaryStatsAvgCalculator(
    column_names=['transaction_amount'],
    timestamp_column_name='timestamp',
    chunk_period='M',
)
avg_calc.fit(reference)
avg_results = avg_calc.calculate(analysis)

avg_df = avg_results.filter(period='analysis').to_df()
avg_df.columns = ['_'.join(c).strip('_') for c in avg_df.columns]
print(avg_df[['chunk_key','transaction_amount_value','transaction_amount_upper_threshold','transaction_amount_lower_threshold','transaction_amount_alert']].to_string(index=False))

alert_row = avg_df[avg_df['transaction_amount_alert'] == True]
alert_avg_transaction_amount = round(float(alert_row['transaction_amount_value'].values[0]), 1)
print("alert_avg_transaction_amount =", alert_avg_transaction_amount)

chunk_key  transaction_amount_value  transaction_amount_upper_threshold  transaction_amount_lower_threshold  transaction_amount_alert
  2018-11               2994.880061                         3021.411123                         2908.434026                     False
  2018-12               2979.182770                         3021.411123                         2908.434026                     False
  2019-01               2993.979414                         3021.411123                         2908.434026                     False
  2019-02               3000.146194                         3021.411123                         2908.434026                     False
  2019-03               2968.669785                         3021.411123                         2908.434026                     False
  2019-04               2923.636529                         3021.411123                         2908.434026                     False
  2019-05               2996.924907                         30